# Synthesising Jettag HGQ2-trained model

Make sure to start this from conda environment with Vitis loaded to path.
`conda activate devenv-hgq`. To install the supplied environment, run `conda env create -f environment-HGQ.yml`. To update env `conda env update -f environment-HGQ.yml`, and remember to restart kernel.

This notebook is used to synthesize the trained models with HLS4MLs backend Vitis Unified. Some synthesis include bitfile-generation, and varies with strategy.

In [1]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
from sklearn.metrics import accuracy_score

# Load Vitis into path
os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']

I0000 00:00:1780921533.228370   19433 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
model_to_test = 'jettag-hgq2'

model_configs = [
    # AdaptiveHP-models
    #{
    #    "description": "AdaptiveHP acc=0.7117 ebops=303 Distributed Arithmetic",
    #    "model_revision": "Training_AdaptiveHP",
    #    "keras_model_path": "jettag-hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7117_ebops=303.keras",
    #    "hls4ml_strategy": "DA",
    #    "hls4ml_generate_bitfile": True,
    #    "hls4ml_revision": "acc=0.7117_ebops=303_VU_DA_bitfile",
    #},
    #{
    #    "description": "AdaptiveHP acc=0.7117 ebops=303 latency",
    #    "model_revision": "Training_AdaptiveHP",
    #    "keras_model_path": "jettag-hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7117_ebops=303.keras",
    #    "hls4ml_strategy": "latency",
    #    "hls4ml_generate_bitfile": True,
    #    "hls4ml_revision": "acc=0.7117_ebops=303_VU_latency_bitfile",
    #},

    #{
    #    "description": "AdaptiveHP acc=0.7426 ebops=1001 Distributed Arithmetic",
    #    "model_revision": "Training_AdaptiveHP",
    #    "keras_model_path": "jettag-hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7426_ebops=1001.keras",
    #    "hls4ml_strategy": "DA",
    #    "hls4ml_generate_bitfile": True,
    #    "hls4ml_revision": "acc=0.7426_ebops=1001_VU_DA_bitfile",
    #},
    {
        "description": "AdaptiveHP acc=0.7426 ebops=1001 AXI Stream",
        "model_revision": "Training_AdaptiveHP",
        "keras_model_path": "jettag-hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7426_ebops=1001.keras",
        "hls4ml_strategy": "latency",
        "hls4ml_generate_bitfile": True,
        "hls4ml_revision": "acc=0.7426_ebops=1001_VU_axistream",
    },
    #{
    #    "description": "AdaptiveHP acc=0.7512 ebops=2895 Distributed Arithmetic",
    #    "model_revision": "Training_AdaptiveHP",
    #    "keras_model_path": "jettag-hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7512_ebops=2895.keras",
    #    "hls4ml_strategy": "DA",
    #    "hls4ml_generate_bitfile": True,
    #    "hls4ml_revision": "acc=0.7512_ebops=2895_VU_DA_bitfile",
    #},
    # FixedHP-models
    #{
    #    "description": "FixedHP acc=0.7590 ebops=11634 Distributed Arithmetic",
    #    "model_revision": "Training_FixedHP",
    #    "keras_model_path": "jettag-hgq2/Training_FixedHP/model_Training_FixedHP_acc=0.7590_ebops=11634.keras",
    #    "hls4ml_strategy": "DA",
    #    "hls4ml_generate_bitfile": True,
    #    "hls4ml_revision": "acc=0.7590_ebops=11634_VU_DA_bitfile",
    #},

    #{
    #    "description": "FixedHP acc=0.7659 ebops=21624 Distributed Arithmetic",
    #    "model_revision": "Training_FixedHP",
    #    "keras_model_path": "jettag-hgq2/Training_FixedHP/model_Training_FixedHP_acc=0.7659_ebops=21624.keras",
    #    "hls4ml_strategy": "DA",
    #    "hls4ml_generate_bitfile": True,
    #    "hls4ml_revision": "acc=0.7659_ebops=21624_VU_DA_bitfile",
    #},
    #{
    #    "description": "FixedHP acc=0.7659 ebops=21624 latency",
    #    "model_revision": "Training_FixedHP",
    #    "keras_model_path": "jettag-hgq2/Training_FixedHP/model_Training_FixedHP_acc=0.7659_ebops=21624.keras",
    #    "hls4ml_strategy": "latency",
    #    "hls4ml_generate_bitfile": True,
    #    "hls4ml_revision": "acc=0.7659_ebops=21624_VU_latency_bitfile",
    #},
]

In [3]:
# Load dataset which is preprocessed in another notebook
# Used for calibration (x) and simulation, which we do not do here
#X_train = np.load("Data/processed_data/X_train.npy")
#X_val = np.load("Data/processed_data/X_val.npy")
x_test = np.load('Data/x_test.npy') # used for calibrating
#y_train = np.load("Data/processed_data/y_train.npy")
#y_val = np.load("Data/processed_data/y_val.npy")
#y_test = np.load("Data/processed_data/y_test.npy")
x_test.dtype


dtype('float32')

In [4]:
import os

def prepare_directory(model_config):
    output_dir = os.path.join(
        os.path.dirname(os.path.abspath(model_config["keras_model_path"])),
        f"hls4ml_prj_{model_config['hls4ml_revision']}",
    )
    os.makedirs(output_dir, exist_ok=True)

    description = f"""
    Description of HLS4ML-project.

    {model_config['description']}

    - Bitfile: {model_config['hls4ml_generate_bitfile']}
    - Environment: devenv-vu-hgq+da (environment-HGQ+DA.yml)
    - Target Device: KV260 (xck26-sfvc784-2LV-c)
    - Dataset: HLS4ML LHC Jets
    - Vivado/Vitis: 2025.2
    - Model Architecture: {model_to_test}
    - Model Revision: {model_config['model_revision']}
    - HLS4ML Revision: {model_config['hls4ml_revision']}

    The model summary is in the parent-directory, `summary.txt`
    """
    with open(os.path.join(output_dir, "description.md"), "w", encoding="utf-8") as f:
        f.write(description)

    return output_dir

In [8]:
from keras.models import load_model
import hgq.layers
import hls4ml
from hgq.utils import trace_minmax

def compile_model(keras_model_path, output_dir, hls4ml_strategy):
    model = load_model(keras_model_path)
    
    # Calibrate datalane in HGQ2-model since it has layers with WRAP
    trace_minmax(model, x_test, verbose=True)
    
    hls_config = hls4ml.utils.config_from_keras_model(model, granularity='name')
    
    strategy = 'Distributed Arithmetic' if hls4ml_strategy == 'DA' else hls4ml_strategy
    hls_config['Model']['Strategy'] = strategy # https://fastmachinelearning.org/hls4ml/api/configuration.html#top-level-configuration

    hls_model = hls4ml.converters.convert_from_keras_model( 
        model,    
        backend     =   'vitisunified',
        hls_config  =   hls_config,
        output_dir  =   output_dir, 
        board       =   'kv260',
        part        =   'xck26-sfvc784-2LV-c',
        clock_period=   '5',
        axi_mode = 'axi_stream',
    )
    hls_model.compile()
    return hls_model

In [9]:
# Hotfix for crashing, see README
os.environ['LD_PRELOAD'] = '/lib/x86_64-linux-gnu/libudev.so.1'

process_model_durations = []

# Run through every model
for i, model_config in enumerate(model_configs):
    print(f"Processing model {i+1}/{len(model_configs)}: {model_config['description']}")
    print(f"%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%")
    start_time = time.time()
    output_dir = prepare_directory(model_config)
    
    hls_model = compile_model(
        model_config["keras_model_path"],
        output_dir,
        model_config["hls4ml_strategy"],
    )

    # Create complete bitfile (Vitis Unified-backend) or IP-block (Vitis-backend)
    hls_model.build(
        synth=True,
        bitfile=model_config["hls4ml_generate_bitfile"],
        csim=False # Simulation (CSIM and COSIM) needs input_data_tb and output_data_tb https://fastmachinelearning.org/hls4ml/autodoc/hls4ml.converters.html#hls4ml.converters.convert_from_keras_model
    )
    
    elapsed_time = time.time() - start_time
    process_model_durations.append({
        'model': model_config['description'],
        'time_seconds': elapsed_time,
        'time_minutes': elapsed_time / 60
    })
    print(f"\nModel {i+1} completed in {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")


Processing model 1/1: AdaptiveHP acc=0.7426 ebops=1001 AXI Stream
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%


dense_0: 455
dense_1: 194
dense_2: 191
dense_3: 161
Total: 1001

****** v++ v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:13
  **** Start of session at: Mon Jun  8 14:33:08 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2025.2 6295257
INFO: [HLS 200-2005] Using work_dir /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/jettag/jettag-hgq2/Training_AdaptiveHP/hls4ml_prj_acc=0.7426_ebops=1001_VU_axistream/vitis_workspace/myproject/vitis_unified_project 
INFO: [HLS 200-2176] Writing Vitis IDE component file /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/jettag/jettag-hgq2/Training_AdaptiveHP/hls4ml_prj_acc=0.7426_ebops=1001_VU_axistream/vitis_workspace/myproject/vitis_unified_project/vitis-comp.json
INFO: [HLS 200-10] Creating and opening component '/home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/jettag/jettag-hgq2/Training_Adapti

In [ ]:
import pandas as pd

timestamp = time.strftime("%Y%m%d_%H%M%S")
timing_df = pd.DataFrame(process_model_durations)
timing_df = timing_df[["model", "time_seconds", "time_minutes"]]
timing_df.to_csv(f"{model_to_test}/timing_summary_{timestamp}.csv", index=False)

display(timing_df)

total_time = timing_df["time_seconds"].sum()
average_time = timing_df["time_seconds"].mean()
print(f"\nTotal time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
print(f"Average time per model: {average_time:.2f} seconds ({average_time/60:.2f} minutes)")

,model,time_seconds,time_minutes
0,AdaptiveHP acc=0.7117 ebops=303 Distributed Ar...,504.187015,8.403117
1,AdaptiveHP acc=0.7426 ebops=1001 Distributed A...,531.031440,8.850524
2,AdaptiveHP acc=0.7512 ebops=2895 Distributed A...,569.839843,9.497331
3,FixedHP acc=0.7590 ebops=11634 Distributed Ari...,697.568978,11.626150
4,FixedHP acc=0.7659 ebops=21624 Distributed Ari...,766.610691,12.776845



Total time: 3069.24 seconds (51.15 minutes)
Average time per model: 613.85 seconds (10.23 minutes)


Copy bitfile and hwh from export-directory to directory for onboard-verification.
Make sure the driver and testdata is copied manually.

In [ ]:
import glob
import shutil

target_dir = '../../onboard-verification/jettag/DUT'

for model_config in model_configs:
    export_dir = os.path.join(
        os.path.dirname(os.path.abspath(model_config["keras_model_path"])),
        f"hls4ml_prj_{model_config['hls4ml_revision']}",
        'export'
    )
    model_target_dir = os.path.join(
        target_dir,
        f"{model_config['model_revision']}_{model_config['hls4ml_revision']}"
    )
    os.makedirs(model_target_dir, exist_ok=True)

    for extension in ('bit', 'hwh'):
        matches = sorted(glob.glob(os.path.join(export_dir, f'*.{extension}')))
        if not matches:
            print(f"ERROR: No .{extension} file found in {export_dir}")
            continue
        shutil.copy2(matches[0], model_target_dir)
        print(f"Copied {matches[0]} -> {model_target_dir}")

Copied /work/development/jettag/jettag-hgq2/Training_AdaptiveHP/hls4ml_prj_acc=0.7117_ebops=303_VU_DA_bitfile/export/system.bit -> ../../onboard-verification/jettag/DUT/Training_AdaptiveHP_acc=0.7117_ebops=303_VU_DA_bitfile
Copied /work/development/jettag/jettag-hgq2/Training_AdaptiveHP/hls4ml_prj_acc=0.7117_ebops=303_VU_DA_bitfile/export/system.hwh -> ../../onboard-verification/jettag/DUT/Training_AdaptiveHP_acc=0.7117_ebops=303_VU_DA_bitfile
Copied /work/development/jettag/jettag-hgq2/Training_AdaptiveHP/hls4ml_prj_acc=0.7426_ebops=1001_VU_DA_bitfile/export/system.bit -> ../../onboard-verification/jettag/DUT/Training_AdaptiveHP_acc=0.7426_ebops=1001_VU_DA_bitfile
Copied /work/development/jettag/jettag-hgq2/Training_AdaptiveHP/hls4ml_prj_acc=0.7426_ebops=1001_VU_DA_bitfile/export/system.hwh -> ../../onboard-verification/jettag/DUT/Training_AdaptiveHP_acc=0.7426_ebops=1001_VU_DA_bitfile
Copied /work/development/jettag/jettag-hgq2/Training_AdaptiveHP/hls4ml_prj_acc=0.7512_ebops=2895_VU_

In [ ]:
# Build and do co-simulation
hls_model.build(
    synth=True, # Only needs to run first time
    csim=True,
    cosim=True,
    
    ) 

NameError: name 'hls_model' is not defined

In [13]:
# Save predicition of a model
from keras.models import load_model
model = "jettag-hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7117_ebops=303.keras"
output_dir = os.path.join(
    os.path.dirname(os.path.abspath(model_configs[0]["keras_model_path"])),
    f"deleteme",
)
os.makedirs(output_dir, exist_ok=True)


model = load_model(model)
y_keras = model.predict(x_test)
np.save("Data/y_303_keras.npy",y_keras)

trace_minmax(model, x_test, verbose=True)
hls_model = hls4ml.converters.convert_from_keras_model( 
    model,    
    backend     =   'vitisunified',
    output_dir  =   output_dir, 
    board       =   'kv260',
    part        =   'xck26-sfvc784-2LV-c',
    clock_period=   '5',
)
hls_model.compile()
y_hls4ml = model.predict(x_test)
np.save("Data/y_303_hls4ml.npy",y_hls4ml)

I0000 00:00:1779794460.948622   91422 service.cc:153] XLA service 0x7f730c03a830 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779794460.948654   91422 service.cc:161]   StreamExecutor [0]: Host, Default Version (Driver: 0.0.0; Runtime: 0.0.0; Toolkit: 0.0.0; DNN: 0.0.0)
I0000 00:00:1779794460.997313   91414 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


1408/5188 ━━━━━━━━━━━━━━━━━━━━ 2s 731us/step

I0000 00:00:1779794461.306416   91414 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


5188/5188 ━━━━━━━━━━━━━━━━━━━━ 2s 369us/step


dense_0: 100
dense_1: 45
dense_2: 72
dense_3: 86
Total: 303
5188/5188 ━━━━━━━━━━━━━━━━━━━━ 1s 114us/step


In [21]:
model.variables

[<Variable path=dense_0/kernel, shape=(16, 64), dtype=float32, value=[[ 1.0050974e+00 -3.8414374e-01 -2.5769362e-01 ...  5.1448941e-01
   -2.6009437e-01  1.1540804e-01]
  [-1.2512651e-01 -1.9938760e-02 -2.7331582e-01 ...  1.3409685e-01
   -1.3621801e-01 -1.2768710e-01]
  [ 1.9701444e-01 -2.5965481e+00 -7.4460095e-01 ... -3.0286840e-01
    1.4698198e-03  5.1417339e-01]
  ...
  [ 7.1849853e-02 -5.2162087e-01 -3.2573062e-01 ... -3.5315636e-01
   -3.5337366e-02 -3.1349939e-01]
  [-1.5916476e+00  2.4584012e+00  2.0121133e+00 ... -1.0092930e+00
    2.3087537e+00 -2.9384062e-01]
  [-3.4404809e+00  3.5179648e+00 -5.0109589e-01 ... -7.3281832e-02
   -7.5435615e-01 -2.9302469e-01]]>,
 <Variable path=dense_0/bias, shape=(64,), dtype=float32, value=[-1.7513621  -0.0052742  -0.16804388 -0.29935807 -0.5613529  -0.6113495
  -1.8445991   0.50681543  0.1110105  -0.87589    -0.9097035   0.754852
  -1.6026405  -0.96419656 -0.9790523  -0.04729041  1.9593952  -0.94155735
  -0.5767358  -0.38507754  0.503308